In [1]:
import os
import functools
import numpy as np
import xarray as xr
from ppe_io import *
from phys import *
import matplotlib.pyplot as plt

In [ ]:
ens_path = '/home/x-yyang40/kangen-wrf/test/ensemble' # change the directory

print(f"Ensemble path exists: {os.path.exists(ens_path)}")
print(f"Contents of parent directory:")

parent_dir = os.path.dirname(ens_path)  # This would be '/home/x-yyang40/WRF-2D/test/ensemble'
print(f"Parent dir: {parent_dir}")
if os.path.exists(parent_dir):
    contents = os.listdir(parent_dir)
    print(f"All contents: {contents}")
    run_dirs = [s for s in contents if s.startswith('run')]
    print(f"Run directories found: {run_dirs}")
else:
    print("Parent directory doesn't exist!")

In [3]:
# from scipy.io import netcdf

# file2read = netcdf.netcdf_file('/home/x-yyang40/WRF-2D/test/ensemble/run0000/wrfout_d01_0001-01-01_00:00:00','r')

# temp = file2read.variables[var] # var can be 'Theta', 'S', 'V', 'U' etc..
# data = temp[:]*1
# file2read.close()

In [4]:

# file2read.close()

In [5]:
@loop_run_paths # (ens_path, operator, name)
def calculate_stat(run_path, operator, name):
    json_path = os.path.join(run_path, JSON_FILENAME)
    result = operator(run_path)
    param = read_json(json_path)
    param[name] = result
    write_json(json_path, param)
    return result

def summarize_json(ens_path):
    @loop_run_paths
    def summarize_json_inner(run_path):
        json_path = os.path.join(run_path, JSON_FILENAME)
        print(f"Checking: {json_path}")  # Debug line
        if not os.path.exists(json_path):
            print(f"JSON file not found: {json_path}")
            return {}
        return read_json(json_path)
    result = summarize_json_inner(ens_path)
    print(f"Result length: {len(result)}")  # Debug line
    if len(result) == 0:
        print("No valid results found!")
        return {}
    keys = list( result[0].keys() )
    return { k: np.array([param[k] for param in result]) for k in keys }

In [6]:
zd = 'bottom_top'
xd = 'west_east'
yd = 'south_north'
hd = (xd, yd)
td = 'Time'
sd = (zd, xd, yd)
def tmax_ssum(da):
    return da.sum(sd).max(td).item(0)

@on_wrfout
def max_liquid(ds):
    return tmax_ssum(ds.QCLOUD + ds.QRAIN)
@on_wrfout
def max_qc(ds):
    return (ds.QCLOUD).max().item(0)
@on_wrfout
def max_qr(ds):
    return (ds.QRAIN).max().item(0)
@on_wrfout
def percentile_incloud(ds, varname='QCLOUD', perc=0.9):
    n_times = len(ds.Time)
    qc_arr = np.sort(ds[varname].transpose('Time',...).data.reshape(n_times,-1), axis=1)
    n_cloud = np.sum(qc_arr > 1e-5, axis=1)
    n_total = qc_arr.shape[1]
    perc_all = 1 - (1-perc)*(n_cloud/n_total)
    index_all = np.floor(n_total * perc_all).astype(np.int_)
    index_all = np.where(index_all >= n_total, n_total-1, index_all)
    return np.max([qc_arr[i,index] for (i, index) in enumerate(index_all)]).item(0)
qc9 = lambda fname: percentile_incloud(fname, varname='QCLOUD', perc=0.9)
@on_wrfout
def max_area(ds):
    return ((ds.QCLOUD + ds.QRAIN).isel(Time=slice(60,None)) > 1e-5).sum(xd).max().item(0) / len(ds.west_east)
@on_wrfout
def total_precip(ds):
    prec = ds.isel(Time=-1).RAINNC.mean().item(0)
    if prec < 1e-4: prec = 0
    return prec
@on_wrfout
def max_w(ds):
    return ds.W.max().item(0)
@on_json
def RH_ml(param):
    qv = param['q_tml'] * 1e-3
    theta0 = param['theta_l0'] - dtheta_l0
    pi_ml = 1 - (g * h_ml) / (func_cp(qv) * theta0)
    T_ml = theta0 * pi_ml
    p_ml = P00 * pi_ml ** (func_cp(qv) / func_R(qv))
    e_ml = func_e(p_ml, qv)
    es_ml = func_esl(T_ml)
    return e_ml / es_ml
@on_json
def with_cloud(param):
    return param['max_area'] > 0

def is_success(run_path):
    try:
        flag = False
        rsl_path = os.path.join(run_path, RSL_OUT_FILENAME)
        with open(rsl_path, 'r') as f:
            rsl_tail = '\n'.join( f.read().split('\n')[-10:] )
            flag = WRF_SUCCESS_STR in rsl_tail
        return flag
    except: return False

In [ ]:
calculate_stat(ens_path, max_liquid, name='max_lm')
calculate_stat(ens_path, is_success, name='success')
calculate_stat(ens_path, max_qc, name='max_qc')
calculate_stat(ens_path, max_qr, name='max_qr')
calculate_stat(ens_path, max_area, name='max_area')
calculate_stat(ens_path, max_w, name='max_w')
calculate_stat(ens_path, with_cloud, name='with_cloud')
calculate_stat(ens_path, total_precip, name='total_precip')
calculate_stat(ens_path, RH_ml, name='RH_ml')
calculate_stat(ens_path, percentile_incloud, name='qc9')

In [ ]:
param = summarize_json(ens_path)

In [ ]:
param['sqrt(L)'] = np.sqrt(param['max_lm'])
keys = ['theta_l0', 'gamma', 'RH_ml', 'h_qt', 'N_m2', 'delt', 'init_rad', 'max_w', 'max_qc', 'max_qr', 'max_area' ]
logscale_keys = ['N_m2', 'total_precip']

n_keys, n_samples = len(keys), len(param['theta_l0'])
gs_kw =  dict(width_ratios=[1]*n_keys, height_ratios=[1]*n_keys)
fig, axd = plt.subplot_mosaic(
    [[f'{ix:d}_{iy:d}' for ix in range(n_keys)] for iy in range(n_keys)],
    gridspec_kw=gs_kw, figsize=(20, 20), layout="constrained" )

fail_mask = ~param['success']
no_cloud_mask = param['success'] & ~param['with_cloud']
success_mask = param['with_cloud']
sample_id = np.arange(n_samples)

sc = None
for ix in range(n_keys):
    for iy in range(n_keys):
        ax = axd[f'{ix:d}_{iy:d}']
        if iy <= ix:
            ax.remove()
        else:
            sc = ax.scatter( param[keys[ix]][success_mask], param[keys[iy]][success_mask],
                             c=sample_id[success_mask], cmap='Spectral', marker='o',
                             edgecolor='k', linewidth=0.5 )
            sc = ax.scatter( param[keys[ix]][no_cloud_mask], param[keys[iy]][no_cloud_mask],
                             c='k', marker='.', s=3)
            sc = ax.scatter( param[keys[ix]][fail_mask], param[keys[iy]][fail_mask],
                             c='k', marker='x')
        if keys[iy] in logscale_keys: ax.set_yscale('log')
        if keys[ix] in logscale_keys: ax.set_xscale('log')
        if iy == n_keys-1: ax.set_xlabel(keys[ix])
        else:              ax.set_xticks([])
        if ix == 0:        ax.set_ylabel(keys[iy])
        else:              ax.set_yticks([])
fig.savefig('ensemble5_parameters.png', dpi=100)

In [ ]:
param = summarize_json(ens_path)
param['h_qt'] /= 1e3
param['RH_ml'] *= 100

botany_range = {
    'theta_l0': (297.5, 299.5),
    'gamma'   : (  4.5,   5.5),
    'q_tml'   : ( 13.5,  15.0),
    'h_qt'    : ( 1200,  2500),
}
logscale_keys = ['N_m2']

n_keys, n_samples = 7, 128
keys  = ['theta_l0', 'gamma', 'RH_ml', 'h_qt', 'N_m2', 'init_rad', 'delt']
names = ['$\\theta_{l0}$ (K)',
         '$\\Gamma$ (K/km)',
         '$\\mathrm{RH}_\\mathrm{ml}$ (%)',
         '$h_{q_t}$ (km)',
         '$N_{m2}$ ($\\mathrm{cm^{-3}}$)',
         '$r_0$ (m)',
         '$\Delta T$ (K)']
mosaic = [[f'{ix:d}_{iy:d}' for ix in range(n_keys)] for iy in range(n_keys)]
gs_kw =  dict(width_ratios=[1]*(n_keys-1)+[0.1], height_ratios=[0.1]+[1]*(n_keys-1))
fig, axd = plt.subplot_mosaic( mosaic, gridspec_kw=gs_kw, figsize=(12, 12), layout="constrained" )
success_mask = param['with_cloud']

sc = None
transform = np.sqrt
for ix in range(n_keys):
    for iy in range(n_keys):
        if not (f'{ix:d}_{iy:d}' in axd): continue
        ax = axd[f'{ix:d}_{iy:d}']
        if iy <= ix:
            ax.remove()
            continue
        _  = ax.scatter( param[keys[ix]][~success_mask], param[keys[iy]][~success_mask],
                         c='k', marker='x', s=15, linewidth=0.8)
        sc = ax.scatter( param[keys[ix]][success_mask], param[keys[iy]][success_mask],
                         c= transform( param['max_qc'][success_mask] ), cmap='viridis', vmax=transform(0.004), vmin=transform(0.0001),
                         marker='o', )
        if keys[iy] in logscale_keys: ax.set_yscale('log')
        if keys[ix] in logscale_keys: ax.set_xscale('log')
        if iy == n_keys-1: ax.set_xlabel(names[ix])
        else:              ax.set_xticks([])
        if ix == 0:        ax.set_ylabel(names[iy])
        else:              ax.set_yticks([])
cb = fig.colorbar(sc, aspect=20, extend='both')
ticks = [0.0001, 0.0002, 0.0005, 0.001, 0.002, 0.003, 0.004]
cb.set_label('Max $q_{<40\\mathrm{μm}}$ [g/kg]')
cb.set_ticks(transform(ticks))
cb.set_ticklabels([f'{tk*1e3:.1f}' for tk in ticks])
# fig.savefig('ensemble_parameters.svg')

In [ ]:
p_width, p_height = 4, 2
n_cases = 8
gs_kw =  dict(width_ratios=[1]*p_width, height_ratios=[1]*p_height)
fig, axd = plt.subplot_mosaic(
    np.arange(p_width*p_height).reshape(p_height, p_width),
    gridspec_kw=gs_kw, figsize=(16, 12), layout="constrained" )

wrfout_sample = xr.open_dataset(os.path.join(ens_path, 'run0000', WRFOUT_FILENAME))
ql_sample = wrfout_sample.isel(south_north=0).QCLOUD.isel(Time=90).data
z_stag = wrfout_sample.PHB.isel(Time=0, south_north=0, west_east=0).data / g
x_stag = np.arange(wrfout_sample.west_east_stag.size) * wrfout_sample.DX
x_stag -= x_stag.mean()
wrfout_sample.close()

pcms = [ axd[i].pcolormesh(x_stag, z_stag, ql_sample, vmin=0, vmax=2e-3)
         for i in axd ]
for i in axd:
    ax = axd[i]
    ax.set_xlim(-1500, 1500)
    ax.set_ylim(0, 4000)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.text(0.05, 0.975, f'{i:03d}', ha='left', va='top', transform=ax.transAxes)

out = []
def update_ax(ens_path, iTime, pcms):
    @loop_run_paths
    @on_wrfout
    def get_data(ds, iTime):
        ql = (ds.QCLOUD + ds.QRAIN).isel(Time=iTime).data
        ql = np.squeeze(ql)
        # print(ql)
        # out.append(ql)
        return ql
    data_list = get_data(ens_path, iTime)
    # data_list = [data_list[i] for i in RH_rank]
    for (pcm, data) in zip(pcms, data_list):
        pcm.set_array(data)

for iTime in range(181):
    print(f'at time {iTime}')
    update_ax(ens_path, iTime, pcms)
    fig.savefig(f'fig/ensemble5.{iTime:04d}.png', dpi=150)

In [ ]:
out

In [ ]:
module load conda
Executing transaction: ...working... done
+---------------------------------------------------------------+
| To use this environment, load the following modules:          |
|     module use $HOME/privatemodules                           |
|     module load conda-env/mypackages-py3.12.8                 |
| (then standard 'conda install' / 'pip install' / run scripts) |
+---------------------------------------------------------------+

module use $HOME/privatemodules
module load conda-env/mypackages-py3.12.8